# ETL

In [30]:
# imports necesarios
import pandas as pd
import os

In [31]:
print("Directorio de trabajo:", os.getcwd())


Directorio de trabajo: c:\PROYECTOS\SQL\homicidios\data


In [32]:
os.chdir('c:\\PROYECTOS\\SQL\\homicidios\\data')

In [33]:
df = pd.read_csv(
    'c:\\PROYECTOS\\SQL\\homicidios\\data\\SAT-HD-BU.csv',
    sep=',',
    encoding='utf-8',
    low_memory=False
)

In [34]:
df.head()

,id_hecho,tipo_persona_id,tipo_persona,cant_inc,cant_vic,federal,provincia_id,provincia_nombre,departamento_id,departamento_nombre,...,victi_situacion_ocupacional_otro,victima_relacion_inculpado,inculpado_sexo,inculpado_identidad_genero,inculpado_identidad_genero_otro,inculpado_tr_edad,inculpado_18_años_o_mas,inculpado_clase,inculpado_otro_clase,inculpado_relacion_victima
0,11535,Víctima idRegistro 12235,Víctima,3,1,No,58,Neuquén,58035,Confluencia,...,NaN,Sin relación,No corresponde,No corresponde,-------,No corresponde,No corresponde,No corresponde,----,No corresponde
1,11535,Imputado idRegistro 12929,Imputado,3,1,No,58,Neuquén,58035,Confluencia,...,---------,No corresponde,Masculino,Varón,NaN,Sin determinar,Sin determinar,Sin determinación,NaN,Sin relación
2,11535,Imputado idRegistro 12931,Imputado,3,1,No,58,Neuquén,58035,Confluencia,...,---------,No corresponde,Masculino,Varón,NaN,Sin determinar,Sin determinar,Sin determinación,NaN,Sin relación
3,11535,Imputado idRegistro 12930,Imputado,3,1,No,58,Neuquén,58035,Confluencia,...,---------,No corresponde,Masculino,Varón,NaN,Sin determinar,Sin determinar,Sin determinación,NaN,Sin relación
4,11536,Imputado idRegistro 12933,Imputado,2,1,No,58,Neuquén,58035,Confluencia,...,---------,No corresponde,Masculino,Varón,NaN,20-24,Sí,Civil,NaN,Sin relación


## Conocer información básica

In [35]:
print('Cantidad de filas y columnas', df.shape)
print('Nombre columnas', df.columns)

Cantidad de filas y columnas (42051, 46)
Nombre columnas Index(['id_hecho', 'tipo_persona_id', 'tipo_persona', 'cant_inc', 'cant_vic',
       'federal', 'provincia_id', 'provincia_nombre', 'departamento_id',
       'departamento_nombre', 'localidad_id', 'localidad_nombre',
       'radio_censal', 'latitud_radio', 'longitud_radio', 'anio', 'mes',
       'fecha_hecho', 'hora_hecho', 'tipo_lugar', 'tipo_lugar_otro',
       'tipo_lugar_ampliado', 'clase_arma', 'clase_arma_otro',
       'en_ocasion_otro_delito', 'en_ocasion_otro_delito_otro',
       'motivo_origen_registro', 'motivo_origen_registro_otro', 'victima_sexo',
       'victima_identidad_genero', 'victima_identidad_genero_otro',
       'victima_tr_edad', 'victima_18_años_o_mas', 'victima_clase',
       'victima_clase_otro', 'victima_situacion_ocupacional',
       'victi_situacion_ocupacional_otro', 'victima_relacion_inculpado',
       'inculpado_sexo', 'inculpado_identidad_genero',
       'inculpado_identidad_genero_otro', 'inculpad

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42051 entries, 0 to 42050
Data columns (total 46 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   id_hecho                          42051 non-null  int64  
 1   tipo_persona_id                   42051 non-null  object 
 2   tipo_persona                      42051 non-null  object 
 3   cant_inc                          42051 non-null  int64  
 4   cant_vic                          42051 non-null  int64  
 5   federal                           42051 non-null  object 
 6   provincia_id                      42051 non-null  int64  
 7   provincia_nombre                  42051 non-null  object 
 8   departamento_id                   42051 non-null  int64  
 9   departamento_nombre               42051 non-null  object 
 10  localidad_id                      42045 non-null  object 
 11  localidad_nombre                  42051 non-null  object 
 12  radi

## Seleccionar solo las variables que voy a utilizar

In [37]:
# selecciona columnas que se van a usar para el análisis
# Selecciona las columnas por su nombre, incluyendo las que faltaban
columnas = [
    'id_hecho',
    'tipo_persona_id',
    'provincia_id',
    'provincia_nombre',
    'departamento_id',          # NUEVA: para desambiguar localidades
    'departamento_nombre',      # NUEVA
    'localidad_id',
    'localidad_nombre',
    'anio',
    'mes',
    'fecha_hecho',
    'hora_hecho',
    'tipo_lugar',
    'tipo_lugar_otro',
    'tipo_lugar_ampliado',
    'motivo_origen_registro',
    'motivo_origen_registro_otro',
    'victima_sexo',
    'victima_identidad_genero',
    'victima_identidad_genero_otro',
    'victima_tr_edad',
    'victima_18_años_o_mas',
    'victima_relacion_inculpado',  # NUEVA: para aproximar femicidios
    'inculpado_sexo',
    'inculpado_identidad_genero',
    'inculpado_identidad_genero_otro',
    'inculpado_tr_edad',
    'inculpado_18_años_o_mas'
]

df_nuevo = df[columnas]

### Filtrar el dataset para conservar únicamente a las víctimas

**El problema:** 
El dataset crudo no registra "un incidente por fila", sino "una persona involucrada por fila". Esto significa que en un mismo homicidio hay filas para las víctimas y filas para los imputados (agresores). Cuando la fila pertenece a un imputado, todas las columnas de la víctima (`victima_sexo`, `victima_tr_edad`, etc.) se rellenan automáticamente con el valor `'No corresponde'`. 

**La solución:** 
Para que nuestro análisis mida el riesgo real y no duplique casos, eliminamos todas las filas donde el sexo de la víctima sea `'No corresponde'`, conservando así solamente a las víctimas reales.

**Nota metodológica importante:** 
Este filtro se debe aplicar **antes** de la etapa de estandarización de textos. Si primero reemplazamos los `'No corresponde'` por `'Sin determinar'` (para limpiar valores nulos), los agresores se camuflarían como "víctimas con datos desconocidos" y sería imposible filtrarlos.

In [38]:
# Filtrar para conservar ÚNICAMENTE a las víctimas reales
# Elimina las filas donde la columna victima_sexo tiene el valor 'No corresponde'
df_nuevo = df_nuevo[df_nuevo['victima_sexo'] != 'No corresponde']

In [39]:
df_nuevo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19089 entries, 0 to 42049
Data columns (total 28 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   id_hecho                         19089 non-null  int64 
 1   tipo_persona_id                  19089 non-null  object
 2   provincia_id                     19089 non-null  int64 
 3   provincia_nombre                 19089 non-null  object
 4   departamento_id                  19089 non-null  int64 
 5   departamento_nombre              19089 non-null  object
 6   localidad_id                     19086 non-null  object
 7   localidad_nombre                 19089 non-null  object
 8   anio                             19089 non-null  int64 
 9   mes                              19089 non-null  int64 
 10  fecha_hecho                      19089 non-null  object
 11  hora_hecho                       19089 non-null  object
 12  tipo_lugar                       1908

## Limpieza

In [40]:
df_nuevo.dtypes

id_hecho                            int64
tipo_persona_id                    object
provincia_id                        int64
provincia_nombre                   object
departamento_id                     int64
departamento_nombre                object
localidad_id                       object
localidad_nombre                   object
anio                                int64
mes                                 int64
fecha_hecho                        object
hora_hecho                         object
tipo_lugar                         object
tipo_lugar_otro                    object
tipo_lugar_ampliado                object
motivo_origen_registro             object
motivo_origen_registro_otro        object
victima_sexo                       object
victima_identidad_genero           object
victima_identidad_genero_otro      object
victima_tr_edad                    object
victima_18_años_o_mas              object
victima_relacion_inculpado         object
inculpado_sexo                    

### Corregir el formato de hora y fecha

In [41]:
# Convertir a datatime (errors='coerse' convierte errores en nulos)
df_nuevo = df_nuevo.copy()
df_nuevo['fecha_hecho'] = pd.to_datetime(df_nuevo['fecha_hecho'], format='%d/%m/%Y', errors='coerce')
df_nuevo['hora_hecho'] = pd.to_datetime(df_nuevo['hora_hecho'],format='%H:%M:%S', errors='coerce').dt.time

### Detectar NaN

In [42]:
# buscar nulos
df_nuevo.isnull().sum()

id_hecho                               0
tipo_persona_id                        0
provincia_id                           0
provincia_nombre                       0
departamento_id                        0
departamento_nombre                    0
localidad_id                           3
localidad_nombre                       0
anio                                   0
mes                                    0
fecha_hecho                            0
hora_hecho                             0
tipo_lugar                             0
tipo_lugar_otro                    19035
tipo_lugar_ampliado                    0
motivo_origen_registro                 0
motivo_origen_registro_otro        19022
victima_sexo                           0
victima_identidad_genero               0
victima_identidad_genero_otro      19089
victima_tr_edad                        0
victima_18_años_o_mas                  0
victima_relacion_inculpado             0
inculpado_sexo                         0
inculpado_identi

In [43]:
# tratar valores nulos y rellenar con texto
df_nuevo = df_nuevo.dropna(subset=['localidad_id']) # no es relevante la cantidad de NaN
df_nuevo['tipo_lugar_otro'] = df_nuevo['tipo_lugar_otro'].fillna('No especificado')
df_nuevo['motivo_origen_registro_otro'] = df_nuevo['motivo_origen_registro_otro'].fillna('No especificado')
df_nuevo['victima_identidad_genero_otro'] = df_nuevo['victima_identidad_genero_otro'].fillna('No especificado')
df_nuevo['inculpado_identidad_genero_otro'] = df_nuevo['inculpado_identidad_genero_otro'].fillna('No especificado')

###  Limpiar y estandarizar textos

In [44]:
# detectar errores de escritura, categorías y desbalances
for col in df_nuevo.select_dtypes(include='object').columns:
    print(df_nuevo[col].value_counts(dropna=False))


tipo_persona_id
Víctima idRegistro 44783    1
Víctima idRegistro 12235    1
Víctima idRegistro 12236    1
Víctima idRegistro 12237    1
Víctima idRegistro 13009    1
                           ..
Víctima idRegistro 12277    1
Víctima idRegistro 12276    1
Víctima idRegistro 12275    1
Víctima idRegistro 26909    1
Víctima idRegistro 12251    1
Name: count, Length: 19086, dtype: int64
provincia_nombre
Buenos Aires                       7818
Santa Fe                           2927
Ciudad Autónoma de Buenos Aires    1027
Córdoba                             982
Tucumán                             929
Mendoza                             748
Salta                               620
Chaco                               573
Misiones                            466
Entre Ríos                          446
Neuquén                             318
Santiago del Estero                 316
Chubut                              316
Corrientes                          301
Formosa                             

### Tratamos 'departamento_id'

In [45]:
# Mostrar los valores únicos de las columnas
df_nuevo['departamento_id'].unique()

array([  58035,   58091,   58077,   58112,   58014,   58056,   58028,
         58007,   66028,   66056,   66126,   66077,   26077,   10049,
         82063,   82042,   82098,   82119,   82084,   82021,   82126,
         82035,   82049,   82091,   26021,   26091,   26035,   26014,
         26007,   66049,   66035,   66112,   10035,   66147,   66133,
         10077,   94014,   78021,   78028,   78014,   70042,   26056,
         70028,   10084,   90084,   34035,   34014,   10098,   70077,
         42002,   42001,   58070,   82105,   82028,   66014,   66098,
         66007,   66140,   82133,   54007,   54063,   54028,   54042,
         54084,   54070,   54119,   54014,   54098,   82007,   54049,
         54056,   70070,   70084,   10112,   82014,   54091,   90119,
         90014,   90007,   90105,   90021,   90063,   10021,   90035,
         82077,   54105,   54021,   54112,   90112,   90098,   90091,
         90028,   90042,   34049,   34056,   34063,   34028,   18112,
         18021,   18

In [46]:
# Cuenta la cantidad de valores únicos, sin contar los valores nulos.
df_nuevo['departamento_id'].nunique()

481

### Tratamos 'victima_identidad_genero'

In [47]:
# ver los valores únicos y las veces que aparecen
df_nuevo['victima_identidad_genero'].value_counts(dropna=False)

victima_identidad_genero
Varón                   15842
Mujer                    3185
Mujer trans/travesti       49
Sin determinar              7
Varón trans                 3
Name: count, dtype: int64

In [48]:
# Clasifica los registros según la identidad de género de la víctima y su vínculo con el inculpado.
# Crea la columna 'categoria_femicidio' como clasificación exploratoria, no como identificación de femicidios confirmados.
import numpy as np

genero = df_nuevo['victima_identidad_genero']
rel = df_nuevo['victima_relacion_inculpado']
es_mujer = genero.isin(['Mujer', 'Mujer trans/travesti'])

condiciones = [
    genero == 'Sin determinar',
    genero.isin(['Varón', 'Varón trans']),
    es_mujer & rel.isin(['Cónyuge/Pareja', 'Ex cónyuge/Ex pareja']),
    es_mujer & rel.isin(['Hijo/a', 'Padre/Madre', 'Hermano/a', 'Otros vínculos familiares']),
    es_mujer & (rel == 'Sin determinación'),
]

categorias = [
    'Identidad de género sin determinar',
    'No aplica',
    'Femicidio íntimo según el vínculo',
    'Posible femicidio familiar',
    'Vínculo sin determinar',
]

df_nuevo['categoria_femicidio'] = np.select(
    condiciones, categorias, default='Otro vínculo'
)

# Contar los registros de cada categoría.
df_nuevo['categoria_femicidio'].value_counts()

categoria_femicidio
No aplica                             15845
Otro vínculo                           1972
Vínculo sin determinar                  763
Posible femicidio familiar              499
Identidad de género sin determinar        7
Name: count, dtype: int64

## Clasificación exploratoria de femicidios

La columna `categoria_femicidio` constituye una clasificación exploratoria basada en la identidad de género de la víctima (`victima_identidad_genero`) y su vínculo con el inculpado (`victima_relacion_inculpado`).

El dataset no identifica femicidios confirmados ni permite determinar si el homicidio fue motivado por razones de género. Por eso, la clasificación se utiliza únicamente como criterio de análisis.

### Criterios de clasificación

| Categoría | Criterio |
|---|---|
| **Femicidio íntimo según el vínculo** | Víctima mujer o mujer trans/travesti, con vínculo `Conyuge/ pareja` o `Ex conyuge / ex pareja`. |
| **Posible femicidio familiar** | Víctima mujer o mujer trans/travesti, con vínculo `Hijo/a`, `Padre/Madre`, `Hermano/a` u `Otros vínculos familiares`. |
| **Vínculo sin determinar** | Víctima mujer o mujer trans/travesti, con vínculo `Sin determinación`. |
| **Otro vínculo** | Víctima mujer o mujer trans/travesti, con vínculo no familiar, comercial o `Sin relación`. |
| **No aplica** | Víctima identificada como `Varón` o `Varón trans`. |
| **Identidad de género sin determinar** | Registros cuya identidad de género figura como `Sin determinar`. |

### Limitaciones y uso

Esta clasificación es una aproximación, no una identificación judicial de femicidios. El vínculo con el inculpado no demuestra por sí solo que exista violencia de género.

Para el análisis principal se utilizará **Femicidio íntimo según el vínculo** como criterio estricto. Como análisis complementario, se incluirá **Posible femicidio familiar** para comparar los resultados.

### Tratamos 'victima_relacion_inculpado'

In [49]:
# ver los valores únicos y las veces que aparecen
df_nuevo['victima_relacion_inculpado'].value_counts(dropna=False)

victima_relacion_inculpado
Sin determinación                 7684
Sin relación                      4863
Otras relaciones no familiares    3114
Conyuge/ pareja                   1257
Otros vínculos familiares          714
Ex conyuge / ex pareja             478
Hijo/a                             339
Padre/Madre                        278
Hermano/a                          278
Empleador                           35
Cliente / proveedor                 20
Socio (Relación comercial)          15
Empleado                            11
Name: count, dtype: int64

In [50]:
# Estandarizar los nombres de las categorías
df_nuevo['victima_relacion_inculpado'] = df_nuevo['victima_relacion_inculpado'].replace({
    'Conyuge/ pareja': 'Cónyuge/Pareja',
    'Ex conyuge / ex pareja': 'Ex cónyuge/Ex pareja',
    'Padre/Madre': 'Padre/Madre',
    'Cliente / proveedor': 'Cliente/Proveedor',
    'Socio (Relación comercial)': 'Socio (relación comercial)'
})

In [51]:
df_nuevo['victima_relacion_inculpado'].value_counts(dropna=False)

victima_relacion_inculpado
Sin determinación                 7684
Sin relación                      4863
Otras relaciones no familiares    3114
Cónyuge/Pareja                    1257
Otros vínculos familiares          714
Ex cónyuge/Ex pareja               478
Hijo/a                             339
Padre/Madre                        278
Hermano/a                          278
Empleador                           35
Cliente/Proveedor                   20
Socio (relación comercial)          15
Empleado                            11
Name: count, dtype: int64

#### Tratamos 'tipo_persona_id'

In [52]:
# Mostrar los valores únicos de las columnas
df_nuevo['tipo_persona_id'].unique()

array(['Víctima idRegistro 12235', 'Víctima idRegistro 12236',
       'Víctima idRegistro 12237', ..., 'Víctima idRegistro 44856',
       'Víctima idRegistro 44857', 'Víctima idRegistro 44783'],
      shape=(19086,), dtype=object)

In [53]:
# Cuenta la cantidad de valores únicos, sin contar los valores nulos.
df_nuevo['tipo_persona_id'].nunique()

19086

In [54]:
# Quitamos el texto y convertimos el ID a número entero
df_nuevo['tipo_persona_id'] = df_nuevo['tipo_persona_id'].str.replace('Víctima idRegistro ', '').astype('int64')

In [55]:
# Verificamos cómo quedó
df_nuevo['tipo_persona_id'].head()

0     12235
6     12236
7     12237
10    13009
12    12239
Name: tipo_persona_id, dtype: int64

In [56]:
df_nuevo['tipo_persona_id'].unique()

array([12235, 12236, 12237, ..., 44856, 44857, 44783], shape=(19086,))

#### Tratamos 'localidad_id'

In [57]:
# Mostrar los valores únicos de las columnas
df_nuevo['localidad_id'].unique()

array(['100', '40', '70', '10', '30', '130', '50', '220', '20', '200',
       '170', '310', '60', '270', '180', '80', '230', '160', '350', '120',
       '110', '105030', '21010', '35020', '210', '90', '21020', '999',
       '140', '260', '150', '450', '390', '400', '250', '190', '240',
       '360', '112020', '35010', '320', '560', '999-999-2', '280', '370',
       '140020', '999-999-6', '290', '300', '28010', '126070', '340',
       '70020', '440', '154040', '77020', '126060', '330', '380', '550',
       '460', '480'], dtype=object)

In [58]:
# Cuenta la cantidad de valores únicos, sin contar los valores nulos.
df_nuevo['localidad_id'].unique()

array(['100', '40', '70', '10', '30', '130', '50', '220', '20', '200',
       '170', '310', '60', '270', '180', '80', '230', '160', '350', '120',
       '110', '105030', '21010', '35020', '210', '90', '21020', '999',
       '140', '260', '150', '450', '390', '400', '250', '190', '240',
       '360', '112020', '35010', '320', '560', '999-999-2', '280', '370',
       '140020', '999-999-6', '290', '300', '28010', '126070', '340',
       '70020', '440', '154040', '77020', '126060', '330', '380', '550',
       '460', '480'], dtype=object)

In [59]:
# Mostrar los valores únicos con su representación exacta
# para ver diferencias entre texto, números, espacios en blanco, etc.
#for x in df_nuevo['localidad_id'].unique():
   # print(repr(x))

In [60]:
# Mostrar las filas donde es igual a, para ver si la info es relevante
print(df_nuevo[df_nuevo['localidad_id'] == '999-999-2'])
print(df_nuevo[df_nuevo['localidad_id'] == '999-999-6'])

       id_hecho  tipo_persona_id  provincia_id  \
6516      16185            17081             2   
15935     21996            23217             2   

                      provincia_nombre  departamento_id  \
6516   Ciudad Autónoma de Buenos Aires             2999   
15935  Ciudad Autónoma de Buenos Aires             2999   

               departamento_nombre localidad_id          localidad_nombre  \
6516   Departamento sin determinar    999-999-2  Localidad sin determinar   
15935  Departamento sin determinar    999-999-2  Localidad sin determinar   

       anio  mes  ... victima_identidad_genero_otro victima_tr_edad  \
6516   2018   11  ...               No especificado           25-29   
15935  2020    8  ...               No especificado           15-19   

      victima_18_años_o_mas victima_relacion_inculpado  inculpado_sexo  \
6516                     Sí               Sin relación  No corresponde   
15935                    No               Sin relación  No corresponde   

  

Se detectaron códigos especiales (999-999-2 y 999-999-6) cuyo contenido en 'localidad_nombre' 
es 'Localidad sin determinar'.
Como el análisis busca mostrar cuáles son las tres provincias y localidades
donde se han detectado la mayor cantidad de homicidios,
se decide reemplazar dichos valores por nulos.

In [61]:
# Reemplazar los códigos de "Localidad sin determinar" por valores nulos (NaN)
df_nuevo['localidad_id'] = df_nuevo['localidad_id'].replace(['999-999-2', '999-999-6'], pd.NA)

In [62]:
# Convertir la columna a tipo numérico (los valores no convertibles quedan como NaN)
df_nuevo['localidad_id'] = pd.to_numeric(df_nuevo['localidad_id'], errors='coerce').astype('Int64')

In [63]:
# Verificar que la conversión se realizó correctamente
df_nuevo['localidad_id'].dtype

Int64Dtype()

In [64]:
df_nuevo['localidad_id'].unique()

<IntegerArray>
[   100,     40,     70,     10,     30,    130,     50,    220,     20,
    200,    170,    310,     60,    270,    180,     80,    230,    160,
    350,    120,    110, 105030,  21010,  35020,    210,     90,  21020,
    999,    140,    260,    150,    450,    390,    400,    250,    190,
    240,    360, 112020,  35010,    320,    560,   <NA>,    280,    370,
 140020,    290,    300,  28010, 126070,    340,  70020,    440, 154040,
  77020, 126060,    330,    380,    550,    460,    480]
Length: 61, dtype: Int64

In [65]:
df_nuevo['localidad_id'].nunique()

60

### Tratamos 'hora_hecho'

In [66]:
# cuenta cuántas veces aparece cada valor de la columna hora_hecho
# ordenándolos de mayor a menor frecuencia.
df_nuevo['hora_hecho'].value_counts()

hora_hecho
11:11:11    6944
11:11:00     408
00:00:00     293
22:00:00     240
21:00:00     226
            ... 
20:31:00       1
14:31:00       1
05:16:00       1
08:29:00       1
02:38:00       1
Name: count, Length: 1152, dtype: int64

In [67]:
# Filtra los registros cuya hora_hecho sea exactamente las 11:11:11.
from datetime import time
df_nuevo[df_nuevo['hora_hecho'] == time(11,11,11)]

,id_hecho,tipo_persona_id,provincia_id,provincia_nombre,departamento_id,departamento_nombre,localidad_id,localidad_nombre,anio,mes,...,victima_identidad_genero_otro,victima_tr_edad,victima_18_años_o_mas,victima_relacion_inculpado,inculpado_sexo,inculpado_identidad_genero,inculpado_identidad_genero_otro,inculpado_tr_edad,inculpado_18_años_o_mas,categoria_femicidio
773,12038,12759,82,Santa Fe,82084,Rosario,270,Rosario,2017,8,...,No especificado,65-69,Sí,Sin determinación,No corresponde,No corresponde,-------,No corresponde,No corresponde,No aplica
1837,12710,13523,82,Santa Fe,82084,Rosario,270,Rosario,2018,1,...,No especificado,85-89,Sí,Hermano/a,No corresponde,No corresponde,-------,No corresponde,No corresponde,No aplica
2222,12955,13802,62,Río Negro,62007,Adolfo Alsina,90,Viedma,2017,1,...,No especificado,35-39,Sí,Sin determinación,No corresponde,No corresponde,-------,No corresponde,No corresponde,No aplica
2224,12956,13803,62,Río Negro,62007,Adolfo Alsina,90,Viedma,2017,1,...,No especificado,15-19,No,Sin determinación,No corresponde,No corresponde,-------,No corresponde,No corresponde,No aplica
2225,12957,13804,62,Río Negro,62042,General Roca,450,General Roca,2017,1,...,No especificado,35-39,Sí,Sin determinación,No corresponde,No corresponde,-------,No corresponde,No corresponde,No aplica
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38101,37743,40007,6,Buenos Aires,6756,San Isidro,10,San Isidro,2024,9,...,No especificado,15-19,No,Otras relaciones no familiares,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica
38103,37744,40008,6,Buenos Aires,6763,San Nicolás,50,San Nicolás de los Arroyos,2024,9,...,No especificado,25-29,Si,Sin determinación,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,Vínculo sin determinar
38107,37745,40009,6,Buenos Aires,6784,Suipacha,20,Suipacha,2024,9,...,No especificado,35-39,Si,Sin determinación,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica
38109,37746,40010,6,Buenos Aires,6805,Tigre,10,Tigre,2024,9,...,No especificado,30-34,Si,Otras relaciones no familiares,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica


In [68]:
df_nuevo[df_nuevo['hora_hecho'] == time(11,11,00)]

,id_hecho,tipo_persona_id,provincia_id,provincia_nombre,departamento_id,departamento_nombre,localidad_id,localidad_nombre,anio,mes,...,victima_identidad_genero_otro,victima_tr_edad,victima_18_años_o_mas,victima_relacion_inculpado,inculpado_sexo,inculpado_identidad_genero,inculpado_identidad_genero_otro,inculpado_tr_edad,inculpado_18_años_o_mas,categoria_femicidio
29817,30728,32791,58,Neuquén,58035,Confluencia,110,Senillosa,2023,8,...,No especificado,25-29,Sí,Sin relación,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica
30070,30904,32969,82,Santa Fe,82084,Rosario,270,Rosario,2023,3,...,No especificado,25-29,Sí,Sin determinación,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,Vínculo sin determinar
30305,31052,33149,82,Santa Fe,82084,Rosario,270,Rosario,2023,7,...,No especificado,4 o menos,No,Hijo/a,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,Posible femicidio familiar
30315,31057,33154,82,Santa Fe,82084,Rosario,270,Rosario,2023,7,...,No especificado,25-29,Sí,Sin determinación,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica
30458,31143,33241,82,Santa Fe,82014,Caseros,50,Casilda,2023,8,...,No especificado,65-69,Sí,Otras relaciones no familiares,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41466,41085,43542,6,Buenos Aires,6658,Quilmes,10,Quilmes,2025,12,...,No especificado,65-69,Si,Otras relaciones no familiares,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica
41468,41086,43543,6,Buenos Aires,6658,Quilmes,10,Quilmes,2025,12,...,No especificado,20-24,Si,Sin relación,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica
41470,41087,43544,6,Buenos Aires,6658,Quilmes,10,Quilmes,2025,12,...,No especificado,30-34,Si,Otras relaciones no familiares,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica
41479,41090,43548,6,Buenos Aires,6658,Quilmes,10,Quilmes,2025,12,...,No especificado,15-19,No,Sin determinación,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica


Se conservan los datos con horas '11:11:11' y '11:11:00' ya que representan una gran cantidad de registros necesarios para el análisis. Sin embargo en caso de analizar la hora del suceso, se procederá a tomar dicha hora dentro de una franja horaria más amplia para evitar el impacto que estos valores puedan tener sobre los resultados.  

### Tratamos 'victima_18_años_o_mas'

In [69]:
# ver los valores únicos y las veces que aparecen
df_nuevo['victima_18_años_o_mas'].value_counts(dropna=False)

victima_18_años_o_mas
Sí                13939
Si                 3300
No                 1520
Sin determinar      327
Name: count, dtype: int64

In [70]:
# renombramos la columna
df_nuevo = df_nuevo.rename(columns={'victima_18_años_o_mas':'victima_18_anios_o_mas'})

In [71]:
# unificamos la categoría 'Si' para evitar duplicados
df_nuevo['victima_18_anios_o_mas'] = df_nuevo['victima_18_anios_o_mas'].replace('Sí','Si')


In [72]:
df_nuevo['victima_18_anios_o_mas'].value_counts()

victima_18_anios_o_mas
Si                17239
No                 1520
Sin determinar      327
Name: count, dtype: int64

### Tratamos 'victima_identidad_genero_otro'

In [73]:
df_nuevo['victima_identidad_genero_otro'].value_counts(dropna=False)

victima_identidad_genero_otro
No especificado    19086
Name: count, dtype: int64

### Tratamos ''inculpado_identidad_genero_otro'

In [74]:
df_nuevo['inculpado_identidad_genero_otro'].value_counts(dropna=False)

inculpado_identidad_genero_otro
-------            13473
No especificado     5613
Name: count, dtype: int64

In [75]:
df_nuevo['inculpado_identidad_genero_otro'] = df_nuevo['inculpado_identidad_genero_otro'].replace(
    '-------','No especificado')

In [76]:
df_nuevo['inculpado_identidad_genero_otro'].value_counts()

inculpado_identidad_genero_otro
No especificado    19086
Name: count, dtype: int64

### Tratamos 'tipo_lugar_ampliado'

In [77]:
df_nuevo['tipo_lugar_ampliado'].value_counts(dropna=False)

tipo_lugar_ampliado
Vía pública                                            10344
Domicilio particular                                    5654
Sin determinar                                          1529
Comercio                                                 411
Interior de rodados                                      275
Campo/descampado/zona rural                              204
Cárcel o comisaría                                       167
Establecimiento zona rural                                71
Otro lugar                                                63
Río/canal/arroyo/mar/dique                                54
Basural/centro de reciclaje/relleno sanitario             40
Establecimiento de salud                                  38
Canchas/club                                              37
Bar/ Boliche/Salón de fiestas/Fiesta                      36
Hotel/Motel/Hospedaje temporario                          30
Casa/edificio abandonado /Obra en construcción            27
Bald

In [78]:
df_nuevo['tipo_lugar_ampliado'] = df_nuevo['tipo_lugar_ampliado'].replace(
    'Bar/ Boliche/Salón de fiestas/Fiesta','Bar/Boliche/Salón de fiestas/Fiesta')

In [79]:
df_nuevo['tipo_lugar_ampliado'] = df_nuevo['tipo_lugar_ampliado'].replace(
    'Casa/edificio abandonado /Obra en construcción','Casa/edificio abandonado/Obra en construcción')

In [80]:
df_nuevo['tipo_lugar_ampliado'].value_counts()

tipo_lugar_ampliado
Vía pública                                            10344
Domicilio particular                                    5654
Sin determinar                                          1529
Comercio                                                 411
Interior de rodados                                      275
Campo/descampado/zona rural                              204
Cárcel o comisaría                                       167
Establecimiento zona rural                                71
Otro lugar                                                63
Río/canal/arroyo/mar/dique                                54
Bar/Boliche/Salón de fiestas/Fiesta                       42
Basural/centro de reciclaje/relleno sanitario             40
Establecimiento de salud                                  38
Canchas/club                                              37
Casa/edificio abandonado/Obra en construcción             33
Hotel/Motel/Hospedaje temporario                          30
Bald

## Duplicados

In [81]:
# contar cuántos duplicados hay
df_nuevo.duplicated().sum()

np.int64(0)

In [82]:
# ver cuáles son
#df_nuevo[df_nuevo.duplicated(keep=False)].head()

In [83]:
#df_nuevo[df_nuevo.duplicated(keep=False)].tail()

In [84]:
# Ver qué pasa con esos casos en el dataframe original (df)
#df.loc[[1333, 1334, 1814, 1816, 41323, 41324,41930, 41932], ['id_hecho', 'tipo_persona_id', 'cant_vic', 'victima_sexo', 'victima_tr_edad']]

En una primera instancia de verificación, nos figuraba 107 duplicados. El codigo comentado arriba, fueron los pasos que se tomaron para verificar y que esos 107 casos marcados por duplicated() no son errores de carga, sino víctimas distintas de homicidios múltiples que coincidían en edad y sexo dentro del mismo hecho.
Se solucionó agregando la columna 'tipo_persona_id', eliminando el texto que tenía y dejandolo solo como valor numérico. 

In [85]:
# eliminamos duplicados
# df_nuevo = df_nuevo.drop_duplicates()

In [86]:
# confirmar que no queden duplicados
# df_nuevo.duplicated().sum()

In [87]:
# ver el tamaño final de tabla
print('Cantidad de filas y columnas', df_nuevo.shape)
print('Nombre columnas', df_nuevo.columns)

Cantidad de filas y columnas (19086, 29)
Nombre columnas Index(['id_hecho', 'tipo_persona_id', 'provincia_id', 'provincia_nombre',
       'departamento_id', 'departamento_nombre', 'localidad_id',
       'localidad_nombre', 'anio', 'mes', 'fecha_hecho', 'hora_hecho',
       'tipo_lugar', 'tipo_lugar_otro', 'tipo_lugar_ampliado',
       'motivo_origen_registro', 'motivo_origen_registro_otro', 'victima_sexo',
       'victima_identidad_genero', 'victima_identidad_genero_otro',
       'victima_tr_edad', 'victima_18_anios_o_mas',
       'victima_relacion_inculpado', 'inculpado_sexo',
       'inculpado_identidad_genero', 'inculpado_identidad_genero_otro',
       'inculpado_tr_edad', 'inculpado_18_años_o_mas', 'categoria_femicidio'],
      dtype='object')


### Tratamos 'victima_tr_edad'

In [88]:
df_nuevo['victima_tr_edad'].value_counts()

victima_tr_edad
20-24             2908
25-29             2776
30-34             2580
15-19             1980
35-39             1804
40-44             1473
45-49             1089
50-54              795
55-59              661
Sin determinar     527
60-64              522
65-69              455
70-74              361
4 o menos          263
75-79              259
10-14              203
80-84              194
5-9                 99
85-89               92
90 y más            45
Name: count, dtype: int64

In [89]:
# Filtra y muestra los registros donde tiene el valor 'Sin determinar'.
df_nuevo[df_nuevo['victima_tr_edad'] == 'Sin determinar']

,id_hecho,tipo_persona_id,provincia_id,provincia_nombre,departamento_id,departamento_nombre,localidad_id,localidad_nombre,anio,mes,...,victima_identidad_genero_otro,victima_tr_edad,victima_18_anios_o_mas,victima_relacion_inculpado,inculpado_sexo,inculpado_identidad_genero,inculpado_identidad_genero_otro,inculpado_tr_edad,inculpado_18_años_o_mas,categoria_femicidio
58,11578,12768,66,Salta,66126,Orán,70,San Ramón de la Nueva Orán (Est. Orán),2017,1,...,No especificado,Sin determinar,Sin determinar,Sin relación,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica
66,11584,12292,26,Chubut,26077,Rawson,40,Trelew,2017,2,...,No especificado,Sin determinar,Si,Sin relación,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica
400,11779,12487,66,Salta,66112,Metán,10,El Galpón (Est. Foguista J. F. Juárez),2017,3,...,No especificado,Sin determinar,Sin determinar,Hijo/a,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,Posible femicidio familiar
576,11921,12639,58,Neuquén,58056,Lácar,10,San Martín de los Andes,2017,8,...,No especificado,Sin determinar,Sin determinar,Sin relación,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica
892,12191,12933,82,Santa Fe,82007,Belgrano,30,Las Parejas,2017,10,...,No especificado,Sin determinar,Sin determinar,Cónyuge/Pareja,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,Otro vínculo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41866,41636,44074,2,Ciudad Autónoma de Buenos Aires,2001,Comuna 1,10,Ciudad de Buenos Aires,2025,12,...,No especificado,Sin determinar,Si,Sin relación,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica
41948,42141,44578,2,Ciudad Autónoma de Buenos Aires,2004,Comuna 4,10,Ciudad de Buenos Aires,2024,12,...,No especificado,Sin determinar,Si,Sin determinación,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica
41958,42147,44584,2,Ciudad Autónoma de Buenos Aires,2006,Comuna 6,10,Ciudad de Buenos Aires,2024,7,...,No especificado,Sin determinar,Si,Sin relación,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica
41976,42156,44593,2,Ciudad Autónoma de Buenos Aires,2008,Comuna 8,10,Ciudad de Buenos Aires,2024,2,...,No especificado,Sin determinar,Si,Sin relación,No corresponde,No corresponde,No especificado,No corresponde,No corresponde,No aplica


### Tratamos 'inculpado_18_años_o_mas'

In [90]:
# renombramos la columna del inculpado
df_nuevo = df_nuevo.rename(columns={'inculpado_18_años_o_mas':'inculpado_18_anios_o_mas'})

In [91]:
# ver los valores únicos y las veces que aparecen
df_nuevo['inculpado_18_anios_o_mas'].value_counts(dropna=False)

inculpado_18_anios_o_mas
No corresponde    19086
Name: count, dtype: int64

### Tratamos 'inculpado_tr_edad'

In [92]:
# ver los valores únicos y las veces que aparecen
df_nuevo['inculpado_tr_edad'].value_counts(dropna=False)

inculpado_tr_edad
No corresponde    19086
Name: count, dtype: int64

In [93]:
df_nuevo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19086 entries, 0 to 42049
Data columns (total 29 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   id_hecho                         19086 non-null  int64         
 1   tipo_persona_id                  19086 non-null  int64         
 2   provincia_id                     19086 non-null  int64         
 3   provincia_nombre                 19086 non-null  object        
 4   departamento_id                  19086 non-null  int64         
 5   departamento_nombre              19086 non-null  object        
 6   localidad_id                     19083 non-null  Int64         
 7   localidad_nombre                 19086 non-null  object        
 8   anio                             19086 non-null  int64         
 9   mes                              19086 non-null  int64         
 10  fecha_hecho                      19086 non-null  datetime64[ns]

In [94]:
df_nuevo.to_csv('homicidios_limpio.csv', index=False, sep=';', encoding='utf-8')